# Unemployment Analysis


**Objective:** Perform exploratory data analysis on unemployment data to uncover regional and temporal trends, with a focus on the impact of the COVID-19 pandemic on unemployment rates in India.

This notebook covers data loading and cleaning, regional analysis, month-wise trends, time-series comparison, top-10 regions by average unemployment, correlation analysis, and a pre-COVID vs. COVID-period comparison.

## 1. Dataset

The analysis uses the Kaggle dataset **“Unemployment in India”**. Keep the downloaded file named `Unemployment_in_India.csv` in the same folder as this notebook before running the code.

The dataset contains fields such as region, date, estimated unemployment rate, estimated employed population, and estimated labour participation rate.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 10

## 2. Load and Inspect the Dataset

The first step is to load the CSV file and inspect its shape, columns, data types, sample records, missing values, and descriptive statistics.

In [ ]:
# Load the dataset
file_path = 'Unemployment_in_India.csv'
df = pd.read_csv(file_path)

# Clean column names and text values
# The Kaggle file can contain leading/trailing spaces in headers and text fields.
df.columns = df.columns.astype(str).str.strip()

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Rename long column names to simpler names
# This makes the later analysis easier to read.
df = df.rename(columns={
    'Estimated Unemployment Rate (%)': 'Unemployment Rate',
    'Estimated Labour Participation Rate (%)': 'Estimated Labour Participation Rate'
})

print('Dataset Shape:', df.shape)
print('\nColumn Names:')
print(df.columns.tolist())

print('\nData Types:')
print(df.dtypes)

print('\nFirst 5 Rows:')
display(df.head())

print('\nDataset Information:')
df.info()

print('\nNull Values:')
print(df.isnull().sum())

print('\nDescriptive Statistics:')
display(df.describe(include='all'))

### Observation

The initial inspection shows the overall structure of the dataset, the available geographic and unemployment-related variables, and whether missing values or inconsistent data types are present. These checks are important before performing any aggregation or visualization.

## 3. Data Cleaning and Type Conversion

The main numeric columns are converted to numeric types safely, the date column is converted to datetime, and rows missing the critical analysis variables are removed. Duplicate rows are also checked.

In [ ]:
# Check duplicate rows
print('Duplicate rows before cleaning:', df.duplicated().sum())

# Convert numeric columns safely
numeric_columns = [
    'Unemployment Rate',
    'Estimated Employed',
    'Estimated Labour Participation Rate'
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert Date to datetime
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(
        df['Date'].astype(str).str.strip(),
        dayfirst=True,
        errors='coerce'
    )

# Remove rows missing critical analysis values
critical_columns = [
    col for col in [
        'Unemployment Rate',
        'Estimated Employed',
        'Estimated Labour Participation Rate',
        'Date'
    ] if col in df.columns
]

df_clean = df.dropna(subset=critical_columns).drop_duplicates().copy()

# Sort by date for time-series analysis
if 'Date' in df_clean.columns:
    df_clean = df_clean.sort_values('Date').reset_index(drop=True)

print('Shape after cleaning:', df_clean.shape)
print('\nMissing values after cleaning:')
print(df_clean.isnull().sum())

print('\nCleaned data types:')
print(df_clean.dtypes)

if 'Date' in df_clean.columns:
    print('\nDate range:', df_clean['Date'].min().date(), 'to', df_clean['Date'].max().date())

### Observation

After cleaning, the dataset is ready for analysis because the key numeric measures have usable numeric types, the date field is available for time-series operations, and rows with missing critical values have been removed.

## 4. Derived Employment Rate

The OASIS task specifically asks for a correlation heatmap involving **unemployment rate, employment rate, and labour participation rate**. The Kaggle dataset provides `Estimated Employed`, but not a column explicitly named `Employment Rate`.

For this notebook, an **Employment Rate (%)** is derived as:

**Employment Rate = Labour Participation Rate × (1 − Unemployment Rate / 100)**

This represents the share of the population represented by the labour force that is employed, expressed on the same percentage scale as the participation and unemployment rates.

In [ ]:
# Derive Employment Rate (%) from labour participation and unemployment rates.
# Example: if labour participation is 45% and unemployment is 10%,
# the implied employment-to-population rate is 45 * (1 - 0.10) = 40.5%.
df_clean['Employment Rate'] = (
    df_clean['Estimated Labour Participation Rate']
    * (1 - df_clean['Unemployment Rate'] / 100)
)

print('Employment Rate created successfully.')
display(df_clean[
    ['Unemployment Rate', 'Employment Rate', 'Estimated Labour Participation Rate']
].head())

### Observation

The derived `Employment Rate` is now available as a percentage measure and can be used directly in the required three-variable correlation analysis.

## 5. Exploratory Data Analysis — Unemployment Rate Statistics

In [ ]:
# Basic unemployment-rate statistics
print('Mean Unemployment Rate:', round(df_clean['Unemployment Rate'].mean(), 2), '%')
print('Median Unemployment Rate:', round(df_clean['Unemployment Rate'].median(), 2), '%')
print('Standard Deviation:', round(df_clean['Unemployment Rate'].std(), 2), '%')
print('Minimum Unemployment Rate:', round(df_clean['Unemployment Rate'].min(), 2), '%')
print('Maximum Unemployment Rate:', round(df_clean['Unemployment Rate'].max(), 2), '%')

# Employment and labour participation statistics
print('\nMean Estimated Employed:', round(df_clean['Estimated Employed'].mean(), 0))
print('Mean Employment Rate:', round(df_clean['Employment Rate'].mean(), 2), '%')
print('Mean Labour Participation Rate:', round(
    df_clean['Estimated Labour Participation Rate'].mean(), 2
), '%')

### Observation

The descriptive statistics provide a baseline for understanding the distribution of unemployment in the dataset. The mean and median indicate the central tendency, while the standard deviation shows how much the unemployment rate varies across observations.

## 6. Region-wise Average Unemployment Rate

The following table and chart summarize the average unemployment rate for each region in the dataset.

In [ ]:
# Identify the geographic column used by the Kaggle dataset
region_col = 'Region' if 'Region' in df_clean.columns else None

if region_col is None:
    raise ValueError('The dataset does not contain a Region column.')

region_avg = (
    df_clean.groupby(region_col)['Unemployment Rate']
    .mean()
    .sort_values(ascending=False)
)

print('Region-wise Average Unemployment Rate:')
display(region_avg.round(2).to_frame('Average Unemployment Rate (%)'))

plt.figure(figsize=(12, 8))
region_avg.sort_values().plot(kind='barh')
plt.xlabel('Average Unemployment Rate (%)')
plt.ylabel('Region')
plt.title('Region-wise Average Unemployment Rate')
plt.tight_layout()
plt.show()

### Observation

The region-wise chart shows clear differences in average unemployment across regions. Some regions record consistently higher average unemployment than others, indicating regional variation in employment conditions within the dataset.

## 7. Month-wise Unemployment Trend

Monthly averages are calculated to identify how unemployment changes over time across all available regions.

In [ ]:
# Calculate monthly average unemployment rate
monthly_data = (
    df_clean
    .groupby(df_clean['Date'].dt.to_period('M'))['Unemployment Rate']
    .agg(['mean', 'std'])
    .reset_index()
)

monthly_data['Date'] = monthly_data['Date'].dt.to_timestamp()

plt.figure(figsize=(14, 6))
plt.plot(
    monthly_data['Date'],
    monthly_data['mean'],
    marker='o',
    linewidth=2,
    label='Monthly Average'
)

# Display one standard-deviation band to show monthly variation across regions.
plt.fill_between(
    monthly_data['Date'],
    monthly_data['mean'] - monthly_data['std'],
    monthly_data['mean'] + monthly_data['std'],
    alpha=0.2,
    label='±1 Standard Deviation'
)

plt.xlabel('Month')
plt.ylabel('Average Unemployment Rate (%)')
plt.title('Month-wise Average Unemployment Rate')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Observation

The monthly trend shows that unemployment does not remain constant over time. Peaks indicate months with comparatively higher average unemployment, while lower points indicate relatively lower unemployment. The shaded band shows variation between regions around the monthly mean.

## 8. Time-series Comparison — Three Regions

To satisfy the time-series requirement, the three regions with the highest average unemployment in the cleaned dataset are tracked over time.

In [ ]:
# Select the three regions with the highest average unemployment rate.
top_3_regions = region_avg.head(3).index.tolist()

plt.figure(figsize=(16, 6))

for region in top_3_regions:
    region_data = df_clean[df_clean[region_col] == region].sort_values('Date')
    plt.plot(
        region_data['Date'],
        region_data['Unemployment Rate'],
        marker='o',
        linewidth=2,
        markersize=4,
        label=str(region)
    )

plt.xlabel('Date')
plt.ylabel('Unemployment Rate (%)')
plt.title('Unemployment Rate Over Time — Three Regions')
plt.xticks(rotation=45)
plt.legend(title='Region')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Regions shown:', ', '.join(map(str, top_3_regions)))

### Observation

The three regional lines allow direct comparison of how unemployment changed over time. The lines can move differently across the same months, showing that regional unemployment patterns were not identical.

## 9. Top 10 Regions with Highest Average Unemployment Rate

In [ ]:
# Top 10 regions by average unemployment rate
# The Kaggle dataset uses the Region column to identify states/regions.
top_10 = region_avg.head(10).sort_values()

plt.figure(figsize=(12, 7))
top_10.plot(kind='barh')

# Add values at the end of each bar
for i, value in enumerate(top_10.values):
    plt.text(value, i, f' {value:.2f}%', va='center')

plt.xlabel('Average Unemployment Rate (%)')
plt.ylabel('Region')
plt.title('Top 10 Regions with Highest Average Unemployment Rate')
plt.tight_layout()
plt.show()

### Observation

The top-10 chart highlights the regions with the highest average unemployment rates in the dataset. It makes the regional differences easier to compare and identifies the regions that appear at the upper end of the unemployment distribution.

## 10. Distribution of Unemployment Rates — Histogram

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(
    df_clean['Unemployment Rate'],
    bins=30,
    edgecolor='black',
    alpha=0.75
)
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Frequency')
plt.title('Distribution of Unemployment Rates')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Observation

The histogram shows how frequently different unemployment-rate values occur. The shape of the distribution helps identify where most observations are concentrated and whether the data contains a wide spread of rates.

## 11. Distribution by Region — Box Plot

In [ ]:
# Order regions by median unemployment rate for easier comparison.
region_order = (
    df_clean.groupby(region_col)['Unemployment Rate']
    .median()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(12, 9))
sns.boxplot(
    data=df_clean,
    y=region_col,
    x='Unemployment Rate',
    order=region_order
)
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Region')
plt.title('Unemployment Rate Distribution by Region')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### Observation

The box plot compares the spread of unemployment rates across regions. Differences in the median, interquartile range, and extreme values indicate that some regions have more variable unemployment rates than others.

## 12. Correlation Heatmap — Required OASIS Metrics

This heatmap uses the three measures specified in the task checklist:

- **Unemployment Rate (%)**
- **Employment Rate (%)**
- **Labour Participation Rate (%)**

The values range from -1 to +1. Positive values indicate a positive linear association, while negative values indicate an inverse linear association.

In [ ]:
heatmap_columns = [
    'Unemployment Rate',
    'Employment Rate',
    'Estimated Labour Participation Rate'
]

corr_matrix = df_clean[heatmap_columns].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Correlation'}
)
plt.title('Correlation Heatmap: Unemployment, Employment and Labour Participation')
plt.tight_layout()
plt.show()

print('Correlation Matrix:')
display(corr_matrix.round(3))

### Observation

The heatmap shows the direction and strength of linear relationships among unemployment rate, employment rate, and labour participation rate. Strong positive or negative values indicate stronger linear association, while values near zero indicate weaker linear association. Correlation by itself does not establish causation.

## 13. Pre-COVID vs. COVID-period Comparison

For this analysis, observations **before March 2020** are treated as the pre-COVID period, while observations **from March 2020 onward** are treated as the COVID-period dataset, matching the split used in the original analysis.

In [ ]:
# Split the dataset at March 1, 2020.
covid_cutoff = pd.Timestamp('2020-03-01')

pre_covid = df_clean[df_clean['Date'] < covid_cutoff].copy()
covid_period = df_clean[df_clean['Date'] >= covid_cutoff].copy()

print('Pre-COVID Period:', pre_covid['Date'].min().date(), 'to', pre_covid['Date'].max().date())
print('COVID Period:', covid_period['Date'].min().date(), 'to', covid_period['Date'].max().date())

comparison_metrics = [
    'Unemployment Rate',
    'Employment Rate',
    'Estimated Labour Participation Rate'
]

comparison_rows = []

for metric in comparison_metrics:
    pre_mean = pre_covid[metric].mean()
    covid_mean = covid_period[metric].mean()
    absolute_change = covid_mean - pre_mean
    percentage_change = (absolute_change / pre_mean * 100) if pre_mean != 0 else np.nan

    comparison_rows.append({
        'Metric': metric,
        'Pre-COVID Mean': pre_mean,
        'COVID-period Mean': covid_mean,
        'Absolute Change': absolute_change,
        'Percentage Change': percentage_change
    })

comparison_df = pd.DataFrame(comparison_rows)

display(
    comparison_df.style.format({
        'Pre-COVID Mean': '{:.2f}',
        'COVID-period Mean': '{:.2f}',
        'Absolute Change': '{:+.2f}',
        'Percentage Change': '{:+.2f}%'
    })
)

### Observation

The comparison table shows how the mean unemployment rate, derived employment rate, and labour participation rate differ between the two periods. This is a descriptive comparison of the dataset and should not by itself be interpreted as proof that COVID-19 caused every observed change.

## 14. COVID-19 Impact Visualization — Overall Unemployment Rate

In [ ]:
pre_rate = pre_covid['Unemployment Rate'].mean()
covid_rate = covid_period['Unemployment Rate'].mean()

periods = ['Pre-COVID\n(Before Mar 2020)', 'COVID Period\n(Mar 2020 onwards)']
values = [pre_rate, covid_rate]

plt.figure(figsize=(9, 6))
bars = plt.bar(periods, values, edgecolor='black', alpha=0.75)

plt.ylabel('Average Unemployment Rate (%)')
plt.title('Average Unemployment Rate: Pre-COVID vs COVID Period')
plt.ylim(0, max(values) * 1.25)
plt.grid(True, alpha=0.3, axis='y')

for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{value:.2f}%',
        ha='center',
        va='bottom',
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

print(f'Pre-COVID average unemployment rate: {pre_rate:.2f}%')
print(f'COVID-period average unemployment rate: {covid_rate:.2f}%')
print(f'Change: {covid_rate - pre_rate:+.2f} percentage points')

### Observation

The bar chart provides a direct visual comparison of the average unemployment rate before March 2020 and from March 2020 onward. The height difference between the bars represents the change in the dataset's average unemployment rate across the two periods.

## 15. State/Region-wise COVID Impact

For each region, the difference between its COVID-period average unemployment rate and its pre-COVID average is calculated when both periods are available.

In [ ]:
state_impact_rows = []

for region in df_clean[region_col].dropna().unique():
    region_pre = pre_covid[pre_covid[region_col] == region]['Unemployment Rate'].mean()
    region_covid = covid_period[covid_period[region_col] == region]['Unemployment Rate'].mean()

    if not pd.isna(region_pre) and not pd.isna(region_covid):
        state_impact_rows.append({
            'Region': region,
            'Pre-COVID Mean': region_pre,
            'COVID-period Mean': region_covid,
            'Change': region_covid - region_pre
        })

impact_df = pd.DataFrame(state_impact_rows).sort_values('Change', ascending=False)

plt.figure(figsize=(14, 9))
colors = np.where(impact_df['Change'] >= 0, '#E74C3C', '#2E86AB')
plt.barh(impact_df['Region'], impact_df['Change'], color=colors, alpha=0.75)
plt.axvline(0, color='black', linewidth=1)
plt.xlabel('Change in Unemployment Rate (percentage points)')
plt.ylabel('Region')
plt.title('Change in Unemployment Rate by Region: COVID Period vs Pre-COVID')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('Regions with the largest positive and negative changes:')
display(impact_df.head(5).round(2))

### Observation

The region-wise COVID comparison shows that the change in unemployment was not identical across regions. Positive values indicate a higher average unemployment rate during the COVID period than before March 2020, while negative values indicate a lower average in the later period.

## 16. Key Insights and Conclusion

This exploratory analysis reveals several patterns in the dataset:

1. **Regional variation:** Average unemployment differs across regions, indicating substantial geographic variation in the recorded unemployment rates.
2. **Temporal variation:** Monthly unemployment rates change over time rather than remaining constant.
3. **Regional time-series differences:** The three selected regions follow different month-to-month patterns.
4. **Correlation structure:** The required heatmap shows how unemployment, employment, and labour participation are linearly associated in the dataset.
5. **COVID-period comparison:** The mean values of the key employment indicators differ between the pre-COVID and COVID-period observations.

### Important interpretation note

These results are descriptive findings from the supplied dataset. Correlation does not prove causation, and the pre-COVID vs. COVID split describes differences between periods rather than independently proving that COVID-19 caused every observed change.

## 17. OASIS Infobyte Task 2 Checklist

| Requirement | Covered in notebook |
|---|---|
| Download and load a suitable unemployment dataset | ✅ |
| Shape inspection | ✅ |
| Null-value check | ✅ |
| Type conversion | ✅ |
| Region-wise average unemployment | ✅ |
| Month-wise trends | ✅ |
| Time-series for at least 3 regions | ✅ |
| Top 10 states/regions with highest average unemployment | ✅ |
| Heatmap: unemployment, employment, labour participation | ✅ |
| Pre-COVID vs COVID-period comparison | ✅ |
| Written observations / Markdown cells between charts | ✅ |
| Clean, commented Jupyter Notebook | ✅ |